In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{Definition} $$

$$ 
(0, 1)^n \times \{0, 1\}^n \to \mathbb{R}, \quad L(p, t) = 
-\frac{1}{N} \sum \Big(t _i \ln(p_i) + (1 - t_i) \ln(1 - p_i) \Big) 
$$

$$ \text{Derivative} $$

$$ 
\frac{dL}{dp_i} = 
-\frac{1}{N} \Big( t_i\frac{1}{p_i} + (1 - t_i)\frac{1}{1 - p_i}(-1) \Big) =
\frac{1}{N} \frac{p_i - t_i}{p_i(1 - p_i)} 
$$

$$ 
\frac{dL}{dp} = 
\frac{1}{N} (p-t) \oslash (p \odot (1-p)) 
$$

$$ \text{Jacobian} $$

$$
J_L(p) =
\nabla _L(p) =
\begin{bmatrix}
    \frac{dL}{dp_1}, &
    \frac{dL}{dp_2},  &
    \cdots, &
    \frac{dL}{dp_n}
\end{bmatrix}
$$

$$ dL = J_L(p) \, dp $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{dp}, dp \right \rangle =
\left \langle \frac{dF}{dL}, dL \right \rangle
$$

$$ \\[2em] $$

$$
\left \langle \frac{dF}{dL}, dL \right \rangle =
\left \langle \frac{dF}{dL}, J_L(p) \, dp \right \rangle =
\left \langle J_L(p) ^\top \, \frac{dF}{dL}, dp \right \rangle \implies
$$

$$ 
\frac{dF}{dp} = 
J_L(p)^\top \, \frac{dF}{dL} = 
\frac{dL}{dp} \odot \frac{dF}{dL} 
$$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore

def _bce(p: tr.Tensor, t: tr.Tensor, reduction = "mean") -> tr.Tensor:
    """
    Computes the binary cross-entropy loss between predictions `p` and targets `t` with `mean` reduction.
    Note: This function assumes that `p` has already been clamped to avoid log(0) issues.
    """

    
    if reduction != "mean":
        assert False, f"Unsupported reduction: {reduction}"
        
    return -tr.mean((t * p.log() + (1 - t) * (1 - p).log()))


def bce(p: tr.Tensor, t: tr.Tensor, reduction = "mean") -> tr.Tensor:
    """
    Computes the binary cross-entropy loss between predictions `p` and targets `t` with `mean` reduction.
    Note: This function clamps `p` to avoid log(0) issues.
    """

    assert tr.all((p >= -1.0) & (p <= +1.0))

    eps = 1e-7
    pc = p.clamp(eps, 1 - eps)
    return _bce(pc, t, reduction)


class BCEFunction(autograd.Function):
    """Custom autograd function for the binary cross-entropy loss."""

    @staticmethod
    def forward(ctx, p: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        eps = 1e-7
        pc = p.clamp(eps, 1 - eps)
        ctx.save_for_backward(pc, t)
        return _bce(pc, t)

    @staticmethod
    def backward(ctx, dF_dL: tr.Tensor) -> tuple[tr.Tensor, None]:        
        (p, t) = ctx.saved_tensors
        S = p.shape[0]
        FO = p.shape[1]
        N = S * FO

        dy_dp = 1 / N * (p - t) / (p * (1 - p))         
        dF_dp = dy_dp * dF_dL

        return (dF_dp, None)
   

class BCE(nn.Module):
    """Custom module for the binary cross-entropy loss."""

    def __init__(self, reduction: str = "mean"):
        super().__init__()

        if reduction != "mean":
            assert False, f"Unsupported reduction: {reduction}"
        
    def forward(self, p: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        return BCEFunction.apply(p, t)

In [ ]:
# Unit tests

def test_gradcheck(samples=10):
    def fn(p, t):
        return BCEFunction.apply(p, t)

    p = tr.rand(samples, 1, dtype=tr.float64, requires_grad=True)
    t = tr.randint(0, 2, (samples, 1)).float().requires_grad_(False)
    assert autograd.gradcheck(fn, (p, t), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(samples):
    p = tr.rand(samples, 1, requires_grad=True)
    t = tr.randint(0, 2, (samples, 1)).float()

    p1 = p.clone().detach().requires_grad_(True)
    L1 = BCE(reduction='mean')(p1, t)
    L1.backward()

    p2 = p.clone().detach().requires_grad_(True)
    L2 = nn.BCELoss(reduction='mean')(p2, t)
    L2.backward()

    assert L1.item() == approx(L2.item())
    assert p1.grad == approx(p2.grad)


if __name__ == "__main__":
    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)